# 11. Imputación como Features Adicionales

Un agujero que arrastramos desde el notebook 2 sin haberlo notado.

Las siete features derivadas que venimos usando —`social_ratio`, `screen_sleep`, `min_per_open` y
compañía— son cocientes y diferencias entre columnas. **Basta que uno de sus insumos falte para que la
derivada sea NaN**, y como cada columna tiene entre 4 % y 19 % de faltantes, el efecto se acumula:

| feature derivada | filas en NaN |
|---|---|
| `ocio_horas` | 30,0 % |
| `social_ratio` | 27,4 % |
| `min_per_open` | 23,4 % |
| `screen_sleep` | 19,1 % |
| `otro_uso` | 13,9 % |

**El 44,6 % de las filas tiene al menos una derivada en NaN.** Todo el trabajo de ingeniería de
features de los notebooks 2 a 9 estuvo ausente en casi la mitad del dataset.

## La idea, y la pregunta abierta

Imputar los valores faltantes desbloquea las derivadas. La pregunta es **cómo**: reemplazando los
valores crudos, o agregando los imputados como columnas adicionales.

Hay un argumento teórico a favor de agregar. Un GBM con manejo nativo de NaN aprende una **dirección
de default por nodo**, que es más expresivo que un único estimador puntual: reemplazar el NaN por una
predicción arrastra la fila hacia el medio de la distribución y borra la información de que faltaba.
Mantener ambas columnas le da al modelo la estructura de faltantes *y* la cobertura completa de los
cocientes.

**Pero el argumento predice que reemplazar debería empeorar, y una prueba preliminar sobre una
submuestra de 40k dio lo contrario**: reemplazar mejoró +0,0009 sobre no imputar, aunque menos que
agregar (+0,0020). O sea que en este dataset el manejo nativo de NaN puede no ser tan superior como
sugiere el argumento, o el efecto de desbloquear los cocientes domina sobre él.

Este notebook mide las tres alternativas sobre los datos completos y los mismos folds, **sin dar por
sentado el signo de ninguna**.

## Por qué imputar sobre train + test no es leakage

Los imputadores se ajustan sobre `train` y `test` **juntos**, 987.671 filas. No interviene el target
en ningún momento: son regresiones de una columna de features contra las otras. Las features de test
están disponibles al momento de predecir; lo único oculto son las etiquetas. Es preprocesamiento
transductivo, no fuga.

> **Tiempo de ejecución: ~1,5 a 2 horas.** La imputación son ~9 minutos (medidos) y se cachea en disco;
> el resto son las tres validaciones cruzadas.

**Métrica:** ROC AUC · **Dataset:** Playground Series S6E8

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import os, time
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["figure.dpi"] = 100

SEED, N_FOLDS = 42, 5

## 1. Carga y Diagnóstico

In [ ]:
train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")
y = train["addicted_label"].values
NTR, NTE = len(train), len(test)

NUMS = ["age", "daily_screen_time_hours", "weekend_screen_time", "social_media_hours",
        "gaming_hours", "work_study_hours", "sleep_hours", "notifications_per_day",
        "app_opens_per_day"]
CATS = ["gender", "stress_level", "academic_work_impact"]
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(NTR), y))


def derivadas_de(d):
    """Las 7 derivadas de los notebooks 2 a 10, sobre las columnas que se le pasen."""
    comp = ["social_media_hours", "gaming_hours", "work_study_hours"]
    return pd.DataFrame({
        "otro_uso":      d["daily_screen_time_hours"] - d[comp].sum(axis=1),
        "ocio_horas":    d["social_media_hours"] + d["gaming_hours"],
        "screen_sleep":  d["daily_screen_time_hours"] / d["sleep_hours"],
        "social_ratio":  d["social_media_hours"] / d["daily_screen_time_hours"],
        "gaming_ratio":  d["gaming_hours"] / d["daily_screen_time_hours"],
        "weekend_ratio": d["weekend_screen_time"] / d["daily_screen_time_hours"],
        "min_per_open":  d["daily_screen_time_hours"] * 60 / d["app_opens_per_day"],
    }).replace([np.inf, -np.inf], np.nan)


DER_CRUDAS = derivadas_de(train)
print("Faltantes en las derivadas calculadas sobre los valores CRUDOS:")
for c in DER_CRUDAS.columns:
    print(f"  {c:16s} {DER_CRUDAS[c].isna().mean():>6.1%}")
print(f"\n  Filas con al menos una derivada en NaN: "
      f"{DER_CRUDAS.isna().any(axis=1).mean():.1%}")

## 2. Imputación Transductiva

Un `XGBRegressor` por columna numérica, ajustado sobre las filas donde esa columna está observada,
usando todas las demás columnas como predictores. Se ajusta sobre `train + test` concatenados.

El resultado se guarda en `../models/`: son ~9 minutos de cómputo que no hace falta repetir si se
vuelve a correr el notebook.

In [ ]:
RUTA_TR = "../models/11_imputado_train.npy"
RUTA_TE = "../models/11_imputado_test.npy"

if os.path.exists(RUTA_TR) and os.path.exists(RUTA_TE):
    IMP_TR = pd.DataFrame(np.load(RUTA_TR), columns=NUMS)
    IMP_TE = pd.DataFrame(np.load(RUTA_TE), columns=NUMS)
    print(f"Imputacion cargada del cache ({RUTA_TR})")
else:
    PARAMS_IMP = dict(n_estimators=400, learning_rate=0.08, max_depth=6, subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=20, tree_method="hist",
                      enable_categorical=True, n_jobs=-1, random_state=SEED)

    full = pd.concat([train[NUMS + CATS], test[NUMS + CATS]], ignore_index=True)
    for c in CATS:
        full[c] = full[c].astype("category")
    salida = full[NUMS].copy()

    t0 = time.time()
    for col in NUMS:
        obs = full[col].notna().values
        feats = [c for c in NUMS + CATS if c != col]
        modelo = XGBRegressor(**PARAMS_IMP).fit(full.loc[obs, feats], full.loc[obs, col])
        if (~obs).sum():
            salida.loc[~obs, col] = modelo.predict(full.loc[~obs, feats])
        print(f"  {col:26s} {(~obs).sum():>7,} imputados | {(time.time()-t0)/60:4.1f} min")

    IMP_TR = salida.iloc[:NTR].reset_index(drop=True).astype(np.float32)
    IMP_TE = salida.iloc[NTR:].reset_index(drop=True).astype(np.float32)
    np.save(RUTA_TR, IMP_TR.values); np.save(RUTA_TE, IMP_TE.values)
    print(f"\nImputacion completa en {(time.time()-t0)/60:.0f} min y cacheada en disco.")

assert IMP_TR.notna().all().all() and IMP_TE.notna().all().all(), "quedaron NaN sin imputar"
print(f"\nSin NaN: {IMP_TR.shape[0]:,} x {IMP_TR.shape[1]} (train), "
      f"{IMP_TE.shape[0]:,} x {IMP_TE.shape[1]} (test)")

## 3. Features sobre los Valores Imputados

Con las columnas completas, las derivadas dejan de tener NaN — y se pueden construir varias más que
antes no tenían sentido porque habrían estado ausentes en la mitad de las filas.

Se agregan además las **banderas de faltante**, una por columna. El notebook 7 midió que la ausencia
acá es MCAR (delta máximo de 0,0042 en la tasa de target contra una base de 0,7094), así que la
expectativa honesta es que no aporten nada. Se incluyen igual, porque son gratis y porque una
predicción explícita que se cumple vale más que una omisión.

In [ ]:
def features_imputadas(imp, crudo):
    """Derivadas sobre valores imputados + banderas de faltante."""
    d = imp
    X = derivadas_de(d).add_suffix("_imp")

    dd, s, g = d["daily_screen_time_hours"], d["social_media_hours"], d["gaming_hours"]
    w, wk, sl = d["work_study_hours"], d["weekend_screen_time"], d["sleep_hours"]
    n, o = d["notifications_per_day"], d["app_opens_per_day"]
    partes = s + g + w

    X["leisure_imp"]       = dd - w
    X["work_frac_imp"]     = w / dd
    X["leisure_frac_imp"]  = (dd - w) / dd
    X["resid_frac_imp"]    = (dd - partes) / dd
    X["week_total_imp"]    = 5 * dd + 2 * wk
    X["awake_frac_imp"]    = dd / (24 - sl)
    X["free_time_imp"]     = 24 - sl - dd - w
    X["notif_per_open_imp"] = n / o

    for c in NUMS:
        X[f"falta_{c}"] = crudo[c].isna().astype(np.float32).values

    return X.replace([np.inf, -np.inf], np.nan).astype(np.float32)


FIMP_TR = features_imputadas(IMP_TR, train)
FIMP_TE = features_imputadas(IMP_TE, test)
nuevas = [c for c in FIMP_TR.columns if c.endswith("_imp")]
banderas = [c for c in FIMP_TR.columns if c.startswith("falta_")]
print(f"{len(nuevas)} features derivadas sobre imputados + {len(banderas)} banderas de faltante")
print(f"NaN restantes en el bloque: {FIMP_TR.isna().mean().mean():.4%}")

## 4. Las Tres Configuraciones

In [ ]:
for col in CATS:
    niveles = sorted(set(train[col].dropna()) | set(test[col].dropna()))
    dt = pd.CategoricalDtype(categories=niveles, ordered=False)
    train[col] = train[col].astype(dt); test[col] = test[col].astype(dt)

# A: lo que veniamos usando — crudas con NaN + derivadas con NaN
A_TR = pd.concat([train[NUMS], DER_CRUDAS, train[CATS]], axis=1)
A_TE = pd.concat([test[NUMS], derivadas_de(test), test[CATS]], axis=1)

# B: REEMPLAZANDO — los valores imputados en lugar de los crudos
B_TR = pd.concat([IMP_TR, derivadas_de(IMP_TR), train[CATS]], axis=1)
B_TE = pd.concat([IMP_TE, derivadas_de(IMP_TE), test[CATS]], axis=1)

# C: AGREGANDO — crudas con NaN + derivadas con NaN + bloque imputado + banderas
C_TR = pd.concat([A_TR, FIMP_TR], axis=1)
C_TE = pd.concat([A_TE, FIMP_TE], axis=1)

for n_, (a, _) in {"A. sin imputar": (A_TR, A_TE), "B. reemplazando": (B_TR, B_TE),
                   "C. agregando": (C_TR, C_TE)}.items():
    print(f"  {n_:<18s} {a.shape[1]:>3d} features | "
          f"{a.isna().any(axis=1).mean():>6.1%} de filas con algun NaN")

In [ ]:
PARAMS = dict(
    objective="binary:logistic", eval_metric="auc", tree_method="hist",
    enable_categorical=True, n_estimators=4000, learning_rate=0.1,
    max_depth=5, min_child_weight=13, subsample=0.8469, colsample_bytree=0.6757,
    reg_lambda=0.4489, reg_alpha=0.001045, gamma=0.5520,
    early_stopping_rounds=100, random_state=SEED, n_jobs=-1,
)


def evaluar(Xtr, Xte, etiqueta):
    t0 = time.time()
    oof = np.zeros(NTR, np.float32); tst = np.zeros(NTE, np.float32); iters = []
    for itr, iva in FOLDS:
        m = XGBClassifier(**PARAMS)
        m.fit(Xtr.iloc[itr], y[itr], eval_set=[(Xtr.iloc[iva], y[iva])], verbose=False)
        oof[iva] = m.predict_proba(Xtr.iloc[iva])[:, 1]
        tst += m.predict_proba(Xte)[:, 1] / N_FOLDS
        iters.append(m.best_iteration)
    auc = roc_auc_score(y, oof)
    print(f"  {etiqueta:<20s} OOF {auc:.6f} | {Xtr.shape[1]:>3d} feat "
          f"| {np.mean(iters):>5.0f} arboles | {(time.time()-t0)/60:.0f} min")
    return auc, oof, tst, m


print(f"{'='*88}")
print("COMPARACION — mismos folds, mismos hiperparametros")
print(f"{'='*88}")
a_A, oof_A, _,     _   = evaluar(A_TR, A_TE, "A. sin imputar")
a_B, oof_B, _,     _   = evaluar(B_TR, B_TE, "B. reemplazando")
a_C, oof_C, tst_C, m_C = evaluar(C_TR, C_TE, "C. agregando")
print(f"{'='*88}")
print(f"  reemplazar (B - A):  {a_B - a_A:+.6f}")
print(f"  agregar    (C - A):  {a_C - a_A:+.6f}")
print(f"  agregar vs reemplazar (C - B): {a_C - a_B:+.6f}")
print(f"{'='*88}")
mejor = max([("A. sin imputar", a_A), ("B. reemplazando", a_B), ("C. agregando", a_C)],
            key=lambda t: t[1])
print(f"  Gana: {mejor[0]}")
if a_B < a_A:
    print("  Reemplazar empeora -> el manejo nativo de NaN vale mas que el estimador puntual.")
else:
    print("  Reemplazar tambien mejora -> desbloquear los cocientes pesa mas que perder")
    print("  la estructura de faltantes. Contradice el argumento teorico de la introduccion.")

## 5. ¿Qué Usa el Modelo?

In [ ]:
imp = pd.Series(m_C.feature_importances_, index=m_C.feature_names_in_).sort_values()
top = imp.tail(20)

fig, ax = plt.subplots(figsize=(11, 8))
colores = ["seagreen" if n.startswith("falta_") else
           "tomato" if n.endswith("_imp") else "steelblue" for n in top.index]
ax.barh(top.index, top.values, color=colores, edgecolor="black")
ax.set_xlabel("Importancia (gain)")
ax.set_title("Top 20 — rojo: derivadas sobre imputados · verde: banderas · azul: originales",
             fontsize=13)
plt.tight_layout()
plt.show()

for pref, nombre, es_sufijo in [("_imp", "derivadas imputadas", True),
                                ("falta_", "banderas de faltante", False)]:
    sel = imp[[n.endswith(pref) if es_sufijo else n.startswith(pref) for n in imp.index]]
    print(f"  {nombre:22s}: {len(sel):>2d} features, {sel.sum()/imp.sum():>6.1%} de la importancia")
print("\n  Si las banderas se llevan ~0%, confirma lo que midio el notebook 7: la ausencia es MCAR.")

## 6. Artefactos para Combinar

El bloque imputado queda cacheado en `../models/` para que un notebook posterior pueda combinarlo con
el target encoding del notebook 10 sin repetir los 9 minutos de imputación. Ambos bloques son
independientes: el encoding actúa sobre los valores exactos, la imputación sobre la cobertura de los
cocientes.

In [ ]:
np.save("../models/11_imputacion_oof.npy",  oof_C)
np.save("../models/11_imputacion_test.npy", tst_C)
submission = pd.DataFrame({"id": test["id"], "addicted_label": tst_C})
submission.to_csv("../submissions/11_imputacion_submission.csv", index=False)

sample = pd.read_csv("../data/sample_submission.csv")
assert list(submission.columns) == list(sample.columns)
assert (submission["id"].values == sample["id"].values).all()
assert submission["addicted_label"].between(0, 1).all()
assert submission["addicted_label"].notna().all()

print("Guardado:")
print(f"  ../models/11_imputado_train.npy      bloque imputado, reutilizable")
print(f"  ../models/11_imputacion_oof.npy      OOF de la config C")
print(f"  ../submissions/11_imputacion_submission.csv  ({len(submission):,} filas)")
print("\n  Chequeos de formato OK. El inventario de 8_envio_final.ipynb los levanta solos.")

## 7. Conclusiones

_Pendiente de completar tras ejecutar el notebook._ Lo que conviene registrar:

- **El signo de B contra A**, que es la pregunta abierta del notebook. La fuente que motivó esta idea
  reportaba que reemplazar **empeora**; nuestra prueba preliminar sobre 40k filas dio lo contrario
  (+0,0009). Con los datos completos se decide. Si empeora, el manejo nativo de NaN de XGBoost es más
  expresivo que un estimador puntual; si mejora, desbloquear los cocientes pesa más que conservar la
  estructura de faltantes. Cualquiera de las dos es un resultado, y conviene no anticipar ninguna.
- **Cuánto aporta C.** La referencia externa reportaba +0,0012 para esta idea.
- **Qué porcentaje de la importancia se llevan las banderas de faltante.** La predicción del notebook 7
  es que sea ~0 porque la ausencia es MCAR. Si se cumple, es una predicción hecha antes de mirar.
- **Si las derivadas sobre imputados desplazan a las originales** en el top de importancia. Serían la
  misma información con cobertura sobre el 44,6 % de filas donde antes faltaba.